# Dataframes
### creating Dataframes

there are 2 ways to do it.

1. From collections

data = [("Alice", 34), ("Bob", 45), ("Cathy", 29)]

(Upload data as list of tuples and create schema)

df = spark.createDataFrame(data, ["name", "age"])

(spark.createDataFrame(data=my_data,schema=my_schema))

df.show()


2. From external files

df = spark.read.option("header", True).csv("/path/to/file.csv")

df.show(5)


## RDD vs Dataframe

|Feature|	RDD|	DataFrame|
|---|---|---|
|Data Type|	No schema, untyped objects|	Schema-aware (columns + datatypes)|
|API Style|	Functional (map, filter)|	Declarative (SQL-like, select, groupBy)|
|Optimization|	No optimizer|	Catalyst + Tungsten|
|Ease of Use|	Verbose|	Concise|
|Use Cases|	Custom processing, unstructured data|	Tabular data, SQL workloads|




In [0]:
display(dbutils.fs.ls("/databricks-datasets"))


In [0]:
display(dbutils.fs.ls("dbfs:/databricks-datasets/flights/"))

In [0]:
sample_df=spark.read\
    .option("header",True)\
    .option("inferSchema",True)\
    .csv("dbfs:/databricks-datasets/flights/departuredelays.csv")

In [0]:
sample_df.show(5)
sample_df.printSchema()
display(sample_df)



#Select
There are multiple ways to select Columns

###1)dataframe.select("columnName").show()

used when just need to select column

###2)dataframe.select(col("columnName")+5).show()

used when needed to apply column wise transformations

###3)dataframe.select(dataframe['columnName]).show()

used during joins



###DISTINCT

In [0]:
sample_df.select("origin","destination").distinct().show()

###FILTER

In [0]:
sample_df.filter(sample_df.delay>120).show()


###Filtering using functions F

pyspark.sql.functions is a module with hundreds of built-in functions for DataFrames.

In [0]:
from pyspark.sql import functions as F
sample_df.filter(F.col("delay")>120).show()


## Common functions inside F

### Column operations
F.col("age")              # reference a column

F.lit(100)                # literal value (constant)

F.when(F.col("age") > 30, "Senior").otherwise("Junior")  # conditional

###Aggregations

F.sum("salary")

F.avg("score")

F.countDistinct("user_id")

###String functions

F.upper("name")

F.substring("city", 1, 3)

### Date/time functions

F.current_date()

F.year("timestamp")


In [0]:
sample_df.groupBy("origin").count().show()



In [0]:
sample_df.show()


In [0]:
sample_df.createGlobalTempView("flights")

Now as already known there are 2 ways to access spark sql here

1. %sql

2. Via python

In [0]:
%sql
select * from global_temp.flights
where delay>0 and origin="ABE"

If you want it to be directly accessible withoug global_temp

then use

### sample_df.createOrReplaceTempView("flights)

then we can query it using flights

### 🌐 createGlobalTempView(viewName) vs createOrReplaceTempView

🧪 createOrReplaceTempView(viewName)

- Scope: Tied to the current SparkSession.

- Lifetime: Disappears when the session ends.

- Usage: Ideal for temporary, session-local SQL queries.

- Access: Only accessible within the notebook or job that created it.

🌐 createGlobalTempView(viewName)

- Scope: Tied to the entire Spark application.

- Lifetime: Persists across multiple SparkSessions until the application ends.

- Usage: Useful when you want to share a view across notebooks or jobs within the same cluster.

- Access: Must be referenced with the global_temp database prefix



In [0]:
%sql
Drop view if exists global_temp.flights
--Drops table

In [0]:

sample_df.createOrReplaceTempView("flights")

In [0]:
spark.sql("select * from flights").show()